# Cloud × LLM: benchmark on a Kaggle T4

This notebook replays the project's vision and audio benchmarks on a free Kaggle T4, the GPU of an
AWS `g4dn.xlarge`. It clones the public repository and runs `scripts/kaggle_t4_bench.sh`.

**Before running anything**, in the right-hand panel, *Session options*:

- *Accelerator*: **GPU T4 x2** (the benchmark uses the first GPU only);
- *Internet*: **on**.

Changing these settings restarts the session. No dataset is needed: the inputs (a generated quote
and a LibriSpeech clip) are part of the repository. Run the cells in order; the benchmark takes
about 20 minutes. Do not use *Save Version*, and stop the session at the end (GPU hours are counted
while it runs).

## 1. Check the GPU and the internet access

In [ ]:
%%bash
set -euo pipefail
nvidia-smi --query-gpu=index,name,driver_version,memory.total,compute_cap --format=csv
nvidia-smi | grep -o 'CUDA Version: [0-9.]*'
curl -sfI https://github.com > /dev/null && echo 'internet ok'
df -h /kaggle/working /tmp | tail -n 2

Expected: two `Tesla T4` lines with `compute_cap` 7.5, and `internet ok`. If `nvidia-smi` is
missing or GitHub does not answer, fix the session options above.

## 2. Run the benchmark

Installs vLLM 0.29.0 (server) and the project's client in two separate environments, then measures
vision (3 × 1, 2 and 4 concurrent pages) and audio (3 × 1, 2, 4 and 8 concurrent transcriptions).
Only the progress lines are shown; the full log is in `/kaggle/working/kaggle-t4-bench.log`.

To replay a single phase, set `PHASES=audio` (or `vision`) before `bash`: the records already in
`/kaggle/working/out` are kept.

In [ ]:
%%bash
set -euo pipefail
rm -rf /tmp/repo
git clone -q --depth 1 https://github.com/Ayoooub222/Cloud_X_LLM.git /tmp/repo
git -C /tmp/repo log --oneline -1
bash /tmp/repo/scripts/kaggle_t4_bench.sh 2>&1 | tee /kaggle/working/kaggle-t4-bench.log | grep -E '^(==|\[ok\]|\[error\])'

Each run prints one `[ok]` or `[error]` line. If a phase fails, its server log is in
`/kaggle/working/out/vllm-<phase>.log` and the error message is stored in each record.

## 3. Follow the progress (optional)

If the output of the previous cell is lost (for example after the browser tab was closed), this
reads the log again. While cell 2 is still running, run it from *File > Open terminal* instead,
without the `%%bash` line.

In [ ]:
%%bash
grep -E '^(==|\[ok\]|\[error\])' /kaggle/working/kaggle-t4-bench.log | tail -n 40
nvidia-smi --query-gpu=index,memory.used,utilization.gpu,power.draw --format=csv,noheader

## 4. Check the result and free the GPU

In [ ]:
%%bash
pkill -f 'vllm serve' || true
ls -lh /kaggle/working/kaggle-t4.tar.gz
echo "runs ok: $(grep -c '^\[ok\]' /kaggle/working/kaggle-t4-bench.log || true)"
echo "runs failed: $(grep -c '^\[error\]' /kaggle/working/kaggle-t4-bench.log || true)"
grep -h -i -E 'attention backend|dtype' /kaggle/working/out/vllm-*.log | head -n 6 || true
nvidia-smi --query-gpu=index,memory.used --format=csv

Expected: 23 runs ok (vision: one warmup and 9 measurements; audio: one warmup and 12
measurements) and 0 failed.

## 5. Download the archive and stop the session

1. In the *Output* panel (or the file browser, under `/kaggle/working`), download
   `kaggle-t4.tar.gz`.
2. *Session* > **Stop session**.
3. On your machine, from the repository root:

```bash
scripts/import_kaggle.sh ~/Downloads/kaggle-t4.tar.gz
python -m src.runner.summarize vision --by-env --tag conc-c1 --tag conc-c2 --tag conc-c4
python -m src.runner.summarize audio --by-env --tag ls66-c1 --tag ls66-c2 --tag ls66-c4 --tag ls66-c8
```

The import never overwrites a record; the raw archive content is kept under
`outputs/kaggle-import/` (not versioned).